  **O que este script faz?**



  1. **Leitura de Dados (Extract):** Conecta ao Google BigQuery e consome dados tratados da camada Silver contendo informações completas sobre filmes da Marvel e DC.



  2. **Modelagem Dimensional (Transform):** Cria tabelas dimensão (Dimensions) e uma tabela fato (Fact Table) seguindo o modelo Star Schema amplamente utilizado em Data Warehouses analíticos.



  3. **Normalização de Entidades:** Organiza informações em entidades separadas como filmes, franquias, equipe técnica, produção e datas, reduzindo redundância de dados.



  4. **Criação de Chaves Substitutas (Surrogate Keys):** Gera IDs únicos para cada dimensão permitindo relacionamentos otimizados entre tabelas.



  5. **Carga Analítica (Load):** Publica todas as tabelas modeladas em uma camada Gold no BigQuery, pronta para dashboards, BI e análises corporativas.

In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Cliente oficial de comunicação com o BigQuery.
from google.oauth2 import service_account  # Responsável pela autenticação via Service Account.

# Manipulação e Análise de Dados
import pandas as pd  # Biblioteca principal para processamento tabular.

In [ ]:
# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Ajustes visuais para facilitar inspeção de tabelas em notebooks e IDEs.

pd.set_option("display.max_columns", None)  # Exibe todas as colunas disponíveis.
pd.set_option("display.max_rows", 10)  # Limita a exibição para evitar excesso visual.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE AUTENTICAÇÃO E CONEXÃO COM O BIGQUERY
# ==============================================================================

# ------------------------------------------------------------------------------
# Credenciais privadas do projeto no Google Cloud Platform.
#
# ATENÇÃO:
# Nunca compartilhar este arquivo em repositórios públicos.
# ------------------------------------------------------------------------------

meu_service_account_file = "../secrets/_______________________.json"

# ID do projeto na Google Cloud.
meu_gcp_project_id = "_______________________"

# Conversão do JSON de credenciais para objeto autenticado.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# Cliente oficial de conexão com o BigQuery.
bigquery_client = bigquery.Client(credentials=gcp_credentials, project=meu_gcp_project_id)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte:
#   Camada Silver do Data Warehouse.
#
# Objetivo:
#   Consumir dataset previamente tratado contendo informações completas
#   sobre filmes da Marvel e DC.
#
# Dataset:
#   silver.complete_dc_marvel
# ------------------------------------------------------------------------------

df_silver = bigquery_client.query("SELECT * FROM silver.complete_dc_marvel").to_dataframe(create_bqstorage_client=False)

In [ ]:
# ==============================================================================
# 5. CRIAÇÃO DA DIMENSÃO FILME (DIM_FILM)
# ==============================================================================

# A dimensão de filmes concentra atributos descritivos relacionados
# às características principais de cada filme.
#
# Objetivo:
#   Evitar redundância e centralizar informações categóricas.

dim_film = (
    df_silver[
        [
            "FILM",
            "GENRE",
            "RATING_MPA",
            "LANGUAGE",
            "BREAK_EVEN",
            "MCU",
            "MALE_FEMALE_LED",
            "PHASE",
        ]
    ]
    .drop_duplicates()
    .sort_values(by="FILM")
    .reset_index(drop=True)
)

# Criação de chave substituta (Surrogate Key).
# IDs artificiais são utilizados em modelagem dimensional
# para melhorar performance e relacionamentos.

dim_film["ID_FILM"] = dim_film.index + 1

# Visualização exploratória da dimensão criada.
dim_film.head(5)

In [ ]:
# ==============================================================================
# 6. CRIAÇÃO DA DIMENSÃO EQUIPE TÉCNICA (DIM_STAFF)
# ==============================================================================

# Esta dimensão centraliza profissionais envolvidos na produção:
#   - diretores,
#   - roteiristas,
#   - atores principais.

dim_staff = (
    df_silver[
        [
            "DIRECTOR",
            "WRITER",
            "STAR",
        ]
    ]
    .drop_duplicates()
    .sort_values(by=["DIRECTOR", "WRITER"])
    .reset_index(drop=True)
)

# Criação da chave substituta da dimensão.

dim_staff["ID_STAFF"] = dim_staff.index + 1

# Visualização exploratória.
dim_staff.head(2)

In [ ]:
# ==============================================================================
# 7. CRIAÇÃO DA DIMENSÃO FRANQUIA (DIM_FRANCHISE)
# ==============================================================================

# Esta dimensão organiza informações sobre:
#   - universo cinematográfico,
#   - famílias/personagens,
#   - franquias.

dim_franchise = (
    df_silver[
        [
            "FRANCHISE",
            "CHARACTER_FAMILY",
        ]
    ]
    .drop_duplicates()
    .sort_values(by=["FRANCHISE", "CHARACTER_FAMILY"])
    .reset_index(drop=True)
)

# Chave substituta da dimensão.

dim_franchise["ID_FRANCHISE"] = dim_franchise.index + 1

# Visualização exploratória.
dim_franchise.head(2)

In [ ]:
# ==============================================================================
# 8. CRIAÇÃO DA DIMENSÃO PRODUÇÃO (DIM_PRODUCTION)
# ==============================================================================

# Esta dimensão reúne informações corporativas e geográficas:
#   - produtoras,
#   - distribuidoras,
#   - país de origem,
#   - local de filmagem.

dim_production = (
    df_silver[
        [
            "PRODUCTION_COMPANY",
            "DISTRIBUTOR",
            "COUNTRY_ORIGIN",
            "FILMING_LOCATION",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by=[
            "PRODUCTION_COMPANY",
            "DISTRIBUTOR",
            "COUNTRY_ORIGIN",
            "FILMING_LOCATION",
        ]
    )
    .reset_index(drop=True)
)

# Criação da chave substituta da dimensão.

dim_production["ID_PRODUCTION"] = dim_production.index + 1

# Visualização exploratória.
dim_production.head(2)

In [ ]:
# ==============================================================================
# 9. CRIAÇÃO DA DIMENSÃO DATA (DIM_DATE)
# ==============================================================================

# Dimensões de tempo são extremamente importantes em Data Warehouses.
#
# Esta tabela permitirá:
#   - análises temporais,
#   - agregações mensais,
#   - análises trimestrais,
#   - comparações anuais.

dim_date = (
    df_silver[
        [
            "US_RELEASE_DATE",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(by="US_RELEASE_DATE")
    .reset_index(drop=True)
)

# Extração de componentes temporais.

dim_date["YEAR"] = dim_date["US_RELEASE_DATE"].dt.year
dim_date["MONTH"] = dim_date["US_RELEASE_DATE"].dt.month
dim_date["QUARTER"] = dim_date["US_RELEASE_DATE"].dt.quarter

# Criação da chave temporal no formato YYYYMMDD.
#
# Exemplo:
#   20250115

dim_date["ID_DATE"] = dim_date["US_RELEASE_DATE"].dt.strftime("%Y%m%d").astype(int)

# Visualização exploratória.
dim_date.head(2)

In [ ]:
# ==============================================================================
# 10. INICIALIZAÇÃO DA TABELA FATO (FACT TABLE)
# ==============================================================================

# A tabela fato armazenará:
#   - métricas quantitativas,
#   - indicadores financeiros,
#   - avaliações,
#   - medidas analíticas.
#
# Ela será conectada às dimensões através das chaves substitutas.

fact_movies = df_silver.copy()

In [ ]:
# ==============================================================================
# 11. RELACIONAMENTO COM A DIMENSÃO FILME
# ==============================================================================

# Associando os IDs da dimensão filme à tabela fato.

fact_movies = fact_movies.merge(
    dim_film,
    on=[
        "FILM",
        "GENRE",
        "RATING_MPA",
        "LANGUAGE",
        "BREAK_EVEN",
        "MCU",
        "MALE_FEMALE_LED",
        "PHASE",
    ],
    how="left",
)

In [ ]:
# ==============================================================================
# 12. RELACIONAMENTO COM A DIMENSÃO EQUIPE TÉCNICA
# ==============================================================================

# Associando os IDs da dimensão staff à tabela fato.

fact_movies = fact_movies.merge(
    dim_staff,
    on=[
        "DIRECTOR",
        "WRITER",
        "STAR",
    ],
    how="left",
)

In [ ]:
# ==============================================================================
# 13. RELACIONAMENTO COM A DIMENSÃO FRANQUIA
# ==============================================================================

# Associando os IDs da dimensão franquia à tabela fato.

fact_movies = fact_movies.merge(
    dim_franchise,
    on=[
        "FRANCHISE",
        "CHARACTER_FAMILY",
    ],
    how="left",
)

In [ ]:
# ==============================================================================
# 14. RELACIONAMENTO COM A DIMENSÃO PRODUÇÃO
# ==============================================================================

# Associando os IDs da dimensão produção à tabela fato.

fact_movies = fact_movies.merge(
    dim_production,
    on=[
        "PRODUCTION_COMPANY",
        "DISTRIBUTOR",
        "COUNTRY_ORIGIN",
        "FILMING_LOCATION",
    ],
    how="left",
)

In [ ]:
# ==============================================================================
# 15. RELACIONAMENTO COM A DIMENSÃO DATA
# ==============================================================================

# Associando os IDs temporais à tabela fato.

fact_movies = fact_movies.merge(
    dim_date,
    on=[
        "US_RELEASE_DATE",
    ],
    how="left",
)

In [ ]:
# ==============================================================================
# 16. SELEÇÃO FINAL DAS MÉTRICAS ANALÍTICAS
# ==============================================================================

# Mantendo apenas:
#   - chaves dimensionais,
#   - indicadores financeiros,
#   - métricas de avaliação,
#   - dados quantitativos.
#
# Estrutura clássica de tabela fato analítica.

fact_movies = fact_movies[
    [
        "ID_FILM",
        "ID_STAFF",
        "ID_FRANCHISE",
        "ID_PRODUCTION",
        "ID_DATE",
        "BUDGET_x",
        "INFLATION_ADJUSTED_BUDGET",
        "GROSS_WORLD_WIDE",
        "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
        "GROSS_US_CANADA",
        "DOMESTIC",
        "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
        "GROSS_OPENING_WEEKEND",
        "GROSS_TO_BUDGET",
        "RATING_IMDB",
        "ROTTEN_TOMATOES_CRITIC_SCORE",
        "VOTE",
        "OSCAR",
        "NOMINATION",
        "WIN",
        "MINUTES",
    ]
]

In [ ]:
# Visualização exploratória da tabela fato final.
fact_movies

In [ ]:
# ==============================================================================
# 17. ORGANIZAÇÃO DAS TABELAS DO DATA WAREHOUSE
# ==============================================================================

# Estrutura contendo todas as tabelas dimensionais
# e a tabela fato que serão carregadas na camada Gold.

minhas_tabelas = {
    "dim_date": dim_date,
    "dim_film": dim_film,
    "dim_franchise": dim_franchise,
    "dim_production": dim_production,
    "dim_staff": dim_staff,
    "fact_movies": fact_movies,
}

In [ ]:
# ==============================================================================
# 18. CARGA FINAL - CAMADA GOLD (LOAD)
# ==============================================================================

# Publicando as tabelas modeladas no BigQuery.
#
# Camada Gold:
#   - altamente refinada,
#   - pronta para BI,
#   - dashboards,
#   - KPIs,
#   - analytics,
#   - consumo executivo.

for key, value in minhas_tabelas.items():

    # Exibe no terminal o nome da tabela sendo enviada.
    print(key)

    # Upload para o BigQuery.
    value.to_gbq(
        project_id=meu_gcp_project_id,
        destination_table=f"gold_star_schema.{key}",
        if_exists="replace",
        credentials=gcp_credentials,
    )